# 00. Zdefiniowanie problemu i hipotez

Ten notebook odpowiada etapowi: zdefiniowanie problemu, pytania badawczego i hipotez.
Nie zmienia bazy danych. Plik `database/NajnowszaWersjaBazy1205.csv` jest traktowany jako gotowy i wyczyszczony zbiór wejściowy.


## Problem badawczy

Analizujemy, kiedy link lub odniesienie między dwoma subredditami ma negatywny sentyment.
Najważniejsza etykieta modelowana w projekcie to `LINK_SENTIMENT`, gdzie `-1` oznacza link negatywny, a `1` link pozytywny.

Dodatkowo baza zawiera `Content_Sentiment`, czyli sentyment treści policzony modelem transformerowym w poprzednim notebooku projektu.
W benchmarku traktujemy go jako Hugging Face baseline, ale głównym celem predykcji pozostaje `LINK_SENTIMENT`.


## Hipotezy

**H1. Eskalacja emocjonalna**

Prawdopodobieństwo wystąpienia linku negatywnego między dwoma subredditami wzrasta, jeśli w poprzednich 24 godzinach wystąpiły między nimi interakcje o wysokim natężeniu słownictwa związanego z gniewem (`LIWC_Anger`).

**H2. Złożoność poznawcza**

Teksty towarzyszące negatywnym linkom mają niższą złożoność językową, na przykład krótsze słowa, mniej spójników logicznych i niższe wskaźniki czytelności, niż teksty w linkach pozytywnych.

**H3. Model multimodalny**

Modele uczenia maszynowego potrafią przewidzieć wystąpienie negatywnego odniesienia między dwiema społecznościami ze skutecznością F1 powyżej 75%, łącząc cechy lingwistyczne, emocjonalne i strukturalne sieci.


## Operacjonalizacja

- H1: tworzymy cechy historii pary subredditów z poprzednich 24 godzin: liczba wcześniejszych interakcji, średni `LIWC_Anger`, maksymalny `LIWC_Anger`.
- H2: porównujemy grupy `LINK_SENTIMENT = -1` i `LINK_SENTIMENT = 1` dla cech złożoności: średnia długość słowa, `LIWC_Conj`, `Automated readability index`, `LIWC_CogMech`, liczba słów.
- H3: porównujemy klasyczne modele ML, cechy tekstowe TF-IDF, cechy lingwistyczne, cechy sieciowe i warianty Hugging Face.

Przy ocenie modeli najważniejszy jest F1 dla klasy negatywnej oraz macro F1, ponieważ zbiór jest silnie niezbalansowany.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "database" / "NajnowszaWersjaBazy1205.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())


PROJECT_ROOT: C:\Users\szymon\projekt_reddit
DATA_PATH exists: True


In [2]:
preview = pd.read_csv(DATA_PATH, nrows=5)
print("Kolumny:", len(preview.columns))
display(preview[["POST_ID", "TIMESTAMP", "SOURCE_SUBREDDIT", "TARGET_SUBREDDIT", "LINK_SENTIMENT", "Content_Sentiment"]])


Kolumny: 95


,POST_ID,TIMESTAMP,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,LINK_SENTIMENT,Content_Sentiment
0,1u4nrps,2013-12-31 16:39:58,leagueoflegends,teamredditteams,1,0
1,1u4sjvs,2013-12-31 17:37:55,nfl,cfb,1,0
2,1u4qkd,2013-12-31 18:18:37,theredlion,soccer,-1,1
3,1u4w7bs,2013-12-31 18:35:44,dogemarket,dogecoin,1,0
4,1u5df2s,2013-12-31 22:27:50,gfycat,india,1,0


## Wniosek po etapie

Zakres badania jest możliwy do wykonania na gotowej bazie. Nie wykonujemy ponownego czyszczenia danych, tylko wykorzystujemy istniejące kolumny tekstowe, LIWC, sentymentowe, czasowe i sieciowe.
